In [1]:
import numpy as np 
import pandas as pd

books = pd.read_csv('clean_data.csv')


In [8]:
books.isna().sum()

isbn13              0
isbn10              0
title               0
authors            32
categories         30
thumbnail         166
description         0
published_year      0
average_rating      0
num_pages           0
ratings_count       0
title_subtitle      0
identifier          0
dtype: int64

In [2]:
cat_count = books['categories'].value_counts()

cat_count[cat_count>50]

categories
Fiction                      2111
Juvenile Fiction              390
Biography & Autobiography     311
History                       207
Literary Criticism            124
Philosophy                    117
Religion                      117
Comics & Graphic Novels       116
Drama                          86
Juvenile Nonfiction            57
Science                        56
Poetry                         51
Name: count, dtype: int64

We want to have a simple categories as fiction and non fiction. Therefore, for categories with value counts more than 50, we define a simple mapping and for anything less than that or with missing values, we use zero shot models to get the categories

In [3]:
category_mapping = {'Fiction' : "Fiction",
 'Juvenile Fiction': "Children's Fiction",
 'Biography & Autobiography': "Nonfiction",
 'History': "Nonfiction",
 'Literary Criticism': "Nonfiction",
 'Philosophy': "Nonfiction",
 'Religion': "Nonfiction",
 'Comics & Graphic Novels': "Fiction",
 'Drama': "Fiction",
 'Juvenile Nonfiction': "Children's Nonfiction",
 'Science': "Nonfiction",
 'Poetry': "Fiction"}

books['simple_categories']= books['categories'].map(category_mapping)
books.head()

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_subtitle,identifier,simple_categories
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,NaN
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...,NaN
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",NaN


In [4]:
# we have to apply zeroshot to the books, which have their simple categories missing. We use isbn13
# as their identifier and then create a df with only

missing_categories = books.loc[books['simple_categories'].isna(), ['description', 'isbn13']].reset_index(drop = True)
missing_categories

,description,isbn13
0,A new 'Christie for Christmas' -- a full-lengt...,9780002261982
1,Lewis' work on the nature of love divides love...,9780006280897
2,"""In The Problem of Pain, C.S. Lewis, one of th...",9780006280934
3,Until Vasco da Gama discovered the sea-route t...,9780006380832
4,A new-cover reissue of the fourth book in the ...,9780006470229
...,...,...
1449,Not only does Nietzsche for Beginners delve in...,9788125026600
1450,"Forster's lively, informed originality and wit...",9788171565641
1451,On A Train Journey Home To North India After L...,9788172235222
1452,This book tells the tale of a man who goes on ...,9788173031014


In [5]:
# Use a pipeline as a high-level helper
import torch
from transformers import pipeline

pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu


In [8]:
expected_categories = ['Fiction', 'Nonfiction']
pipe(missing_categories['description'][0], expected_categories)

{'sequence': "A new 'Christie for Christmas' -- a full-length novel adapted from her acclaimed play by Charles Osborne Following BLACK COFFEE and THE UNEXPECTED GUEST comes the final Agatha Christie play novelisation, bringing her superb storytelling to a new legion of fans. Clarissa, the wife of a Foreign Office diplomat, is given to daydreaming. 'Supposing I were to come down one morning and find a dead body in the library, what should I do?' she muses. Clarissa has her chance to find out when she discovers a body in the drawing-room of her house in Kent. Desperate to dispose of the body before her husband comes home with an important foreign politician, Clarissa persuades her three house guests to become accessories and accomplices. It seems that the murdered man was not unknown to certain members of the house party (but which ones?), and the search begins for the murderer and the motive, while at the same time trying to persuade a police inspector that there has been no murder at a

In [14]:
def generate_labels(sequence, categories):
    prediction = pipe(sequence, categories)
    max_id = np.argmax(prediction['scores'])
    max_label = prediction['labels'][max_id]
    return max_label

In [16]:
from tqdm import tqdm
tqdm.pandas()

missing_categories['predicted_category'] = missing_categories['description'].progress_apply(lambda x: generate_labels(x, categories=expected_categories))

100%|██████████| 1454/1454 [11:24<00:00,  2.12it/s]


In [17]:
books = books.merge(missing_categories, on='isbn13', how = 'left')

In [22]:
books['simple_categories'] = np.where(books['simple_categories'].isna(), books['predicted_category'], books['simple_categories'])

In [ ]:
books = books.drop(columns=['predicted_category', 'description_y'], axis=1)

In [27]:
books['simple_categories'].value_counts(normalize=True)

simple_categories
Fiction                  0.540312
Nonfiction               0.373677
Children's Fiction       0.075043
Children's Nonfiction    0.010968
Name: proportion, dtype: float64

In [28]:
books.to_csv('books_with_categories.csv', index=False)